# Lab 2 — Compose Recommend + Search into a Multi-Deployment Service (AI-assisted)

**Time:** ~15–20 min  
**Mode:** AI-assisted.

## Where this picks up
Session 3 shows three Serve refactors: a single `IngressAndRecommender` deployment;
then splitting that into `Ingress`, `Recommender`, `DatabaseFacade`; then adding a
`SemanticSearch` deployment behind the same ingress.

In this lab you start with a **single combined** deployment (much like the lecture's
starting point) and drive an AI assistant to refactor it into the **multi-deployment**
shape. You will then add **input validation** with Pydantic and a **/health** route.

## Learning objectives
1. Use `bind` composition: pass one bound deployment into another's `bind(...)`.
2. Drive an AI assistant through a non-trivial Serve refactor and verify each step.
3. Add request validation and a health endpoint without breaking existing callers.

## Setup — the single-deployment starter

In [ ]:
import ray, json, requests
from ray import serve
from starlette.requests import Request

if not ray.is_initialized():
    ray.init()

# Stand-ins for the real Recommend / SemanticSearch logic, so the lab is self-contained.
# Replace these with the real `recommend.Recommend` etc. when wiring against the
# course datasets.
class FakeRecommender:
    def __init__(self, k):
        self.k = k
    def for_user(self, uid):
        return [{'item_id': i, 'name': f'item-{i}', 'price': 9.99} for i in range(self.k)]

class FakeSearcher:
    def search(self, query, k):
        return [{'item_id': i, 'name': f'{query}-{i}', 'price': 4.99} for i in range(k)]

In [ ]:
@serve.deployment
class IngressAndEverything:
    def __init__(self, recos_per_user: int):
        self.rec = FakeRecommender(recos_per_user)
        self.search = FakeSearcher()

    async def __call__(self, request: Request):
        body = json.loads(await request.json())
        out = {'recommendations': self.rec.for_user(body['id'])}
        if 'query' in body:
            out['search_results'] = self.search.search(body['query'], 5)
        return out

serve.run(IngressAndEverything.bind(3))
print(requests.post('http://localhost:8000/', json='{"id": "u-1", "query": "steel bowls"}').json())
serve.shutdown()

## Exercise — drive the refactor

### Prompt A — split into three deployments

> *"Refactor `IngressAndEverything` into three `@serve.deployment`s:
> - `Ingress` — handles the HTTP request, parses the JSON body, and routes
> - `Recommender` — wraps `FakeRecommender`, exposes a `get_recos(user_id)` method
> - `SearchService` — wraps `FakeSearcher`, exposes a `search(query, k)` method
>
> Use `bind` composition so the `Ingress` deployment receives bound handles to the
> other two. The HTTP API must remain identical: same body, same response shape."*

**Acceptance criteria for A:**
- A request `'{"id": "u-1"}'` returns just `{"recommendations": [...]}`
- A request `'{"id": "u-1", "query": "steel bowls"}'` returns both keys.
- The Ingress only calls `await self.recommender.get_recos.remote(...)` (and
  similarly for search) — it does **not** instantiate `FakeRecommender` directly.

In [ ]:
import json
import requests
from starlette.requests import Request
from ray import serve
from ray.serve.handle import DeploymentHandle


@serve.deployment
class Recommender:
    def __init__(self, recos_per_user: int):
        self._rec = FakeRecommender(recos_per_user)

    def get_recos(self, user_id: str):
        return self._rec.for_user(user_id)


@serve.deployment
class SearchService:
    def __init__(self):
        self._search = FakeSearcher()

    def search(self, query: str, k: int):
        return self._search.search(query, k)


@serve.deployment
class Ingress:
    def __init__(
        self,
        recommender: DeploymentHandle,
        search_service: DeploymentHandle,
    ):
        self._recommender = recommender
        self._search_service = search_service

    async def __call__(self, request: Request):
        body = json.loads(await request.json())

        # Kick off the recos call; await its result.
        out = {
            "recommendations": await self._recommender.get_recos.remote(body["id"])
        }

        if "query" in body:
            out["search_results"] = await self._search_service.search.remote(
                body["query"], 5
            )

        return out


# Compose with bind: Ingress receives handles to the other two deployments.
app = Ingress.bind(
    recommender=Recommender.bind(3),
    search_service=SearchService.bind(),
)

serve.run(app)
print(
    requests.post(
        "http://localhost:8000/",
        json='{"id": "u-1"}',
    ).json()
)
print(
    requests.post(
        "http://localhost:8000/",
        json='{"id": "u-1", "query": "steel bowls"}',
    ).json()
)
serve.shutdown()

### Prompt B — add Pydantic input validation

> *"Add a Pydantic model `SearchRecRequest` with fields `id: str` (required)
> and `query: str | None`. Validate the request body inside `Ingress.__call__`
> and return a 400 response with a useful message when validation fails."*

Validate yourself that:
- POST with empty body → 400.
- POST with `{"query": "hi"}` (no id) → 400.
- POST with `{"id": 123}` (wrong type) → 400.
- POST with `{"id": "u-1"}` → 200.

In [ ]:
import json
import requests
from pydantic import BaseModel, ValidationError
from starlette.requests import Request
from starlette.responses import JSONResponse
from ray import serve
from ray.serve.handle import DeploymentHandle


class SearchRecRequest(BaseModel):
    id: str
    query: str | None = None


@serve.deployment
class Recommender:
    def __init__(self, recos_per_user: int):
        self._rec = FakeRecommender(recos_per_user)

    def get_recos(self, user_id: str):
        return self._rec.for_user(user_id)


@serve.deployment
class SearchService:
    def __init__(self):
        self._search = FakeSearcher()

    def search(self, query: str, k: int):
        return self._search.search(query, k)


@serve.deployment
class Ingress:
    def __init__(
        self,
        recommender: DeploymentHandle,
        search_service: DeploymentHandle,
    ):
        self._recommender = recommender
        self._search_service = search_service

    async def __call__(self, request: Request):
        # Parse + validate. The original code did json.loads(await request.json()),
        # which double-decodes — request.json() already returns a parsed object.
        # We accept either a dict or a JSON string for compatibility with the
        # original client that sends json='{"id": "u-1", ...}'.
        raw = await request.json()
        if isinstance(raw, str):
            try:
                raw = json.loads(raw)
            except json.JSONDecodeError as e:
                return JSONResponse(
                    {"error": "invalid JSON", "detail": str(e)},
                    status_code=400,
                )

        try:
            body = SearchRecRequest.model_validate(raw)
        except ValidationError as e:
            return JSONResponse(
                {"error": "invalid request body", "detail": e.errors()},
                status_code=400,
            )

        out = {"recommendations": await self._recommender.get_recos.remote(body.id)}

        if body.query is not None:
            out["search_results"] = await self._search_service.search.remote(
                body.query, 5
            )

        return out


app = Ingress.bind(
    recommender=Recommender.bind(3),
    search_service=SearchService.bind(),
)

serve.run(app)

Now we'll check:
```
POST with empty body → 400.
POST with {"query": "hi"} (no id) → 400.
POST with {"id": 123} (wrong type) → 400.
POST with {"id": "u-1"} → 200.
```

In [ ]:
requests.post("http://localhost:8000/", json='')

In [ ]:
requests.post("http://localhost:8000/", json='{"query": "hi"}' )

In [ ]:
requests.post("http://localhost:8000/", json='{"id": 123}' )

In [ ]:
requests.post("http://localhost:8000/", json='{"id": "u-1"}' )

In [ ]:
print(
    requests.post(
        "http://localhost:8000/",
        json='{"id": "u-1", "query": "steel bowls"}',
    ).json()
)
serve.shutdown()

### Prompt C — add a `/health` route

> *"Add a `/health` GET route on the `Ingress` deployment that returns
> `{"status": "ok"}` and does *not* call the downstream deployments.
> Use FastAPI ingress integration (`@serve.ingress(app)`) — that's the cleanest
> way to add a second route."*

Confirm that `requests.get('http://localhost:8000/health')` returns 200 with
`{"status": "ok"}` *and* the existing POST behavior still works.

In [ ]:
import json
import requests
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from ray import serve
from ray.serve.handle import DeploymentHandle


class SearchRecRequest(BaseModel):
    id: str
    query: str | None = None


@serve.deployment
class Recommender:
    def __init__(self, recos_per_user: int):
        self._rec = FakeRecommender(recos_per_user)

    def get_recos(self, user_id: str):
        return self._rec.for_user(user_id)


@serve.deployment
class SearchService:
    def __init__(self):
        self._search = FakeSearcher()

    def search(self, query: str, k: int):
        return self._search.search(query, k)


api = FastAPI()


@serve.deployment
@serve.ingress(api)
class Ingress:
    def __init__(
        self,
        recommender: DeploymentHandle,
        search_service: DeploymentHandle,
    ):
        self._recommender = recommender
        self._search_service = search_service

    @api.get("/health")
    async def health(self):
        return {"status": "ok"}

    @api.post("/")
    async def handle(self, body: SearchRecRequest):
        out = {"recommendations": await self._recommender.get_recos.remote(body.id)}
        if body.query is not None:
            out["search_results"] = await self._search_service.search.remote(
                body.query, 5
            )
        return out


app = Ingress.bind(
    recommender=Recommender.bind(3),
    search_service=SearchService.bind(),
)

serve.run(app)
print(
    requests.post(
        "http://localhost:8000/",
        json={"id": "u-1", "query": "steel bowls"},
    ).json()
)
print(requests.get("http://localhost:8000/health").json())
serve.shutdown()

---

# Reference solution

*One reasonable target. Compare with the lecture's `search_and_recommend.py`.*

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ValidationError

class SearchRecRequest(BaseModel):
    id: str
    query: str | None = None

app = FastAPI()

@serve.deployment
class Recommender:
    def __init__(self, k):
        self._inner = FakeRecommender(k)
    async def get_recos(self, user_id):
        return self._inner.for_user(user_id)

@serve.deployment
class SearchService:
    def __init__(self):
        self._inner = FakeSearcher()
    async def search(self, query, k):
        return self._inner.search(query, k)

@serve.deployment
@serve.ingress(app)
class Ingress:
    def __init__(self, recommender, search):
        self.rec = recommender
        self.search = search

    @app.get('/health')
    async def health(self):
        return {'status': 'ok'}

    @app.post('/')
    async def root(self, body: SearchRecRequest):
        out = {'recommendations': await self.rec.get_recos.remote(body.id)}
        if body.query:
            out['search_results'] = await self.search.search.remote(body.query, 5)
        return out

rec_b    = Recommender.bind(3)
search_b = SearchService.bind()
serve.run(Ingress.bind(rec_b, search_b))

print(requests.get('http://localhost:8000/health').json())
print(requests.post('http://localhost:8000/', json={'id': 'u-1'}).json())
print(requests.post('http://localhost:8000/', json={'id': 'u-1', 'query': 'steel bowls'}).json())
print(requests.post('http://localhost:8000/', json={'query': 'oops no id'}).status_code)
serve.shutdown()